In [ ]:
import pandas as pd
from google.colab import files

# 1. Configuramos para ver todas las filas sin recortar
pd.set_option('display.max_rows', None)

print("Por favor, selecciona tu archivo Excel (.xlsx):")
subido = files.upload()

# Obtenemos automáticamente el nombre del archivo que acabas de elegir
nombre_archivo = list(subido.keys())[0]

# 2. Leemos TODAS las pestañas del archivo Excel que seleccionaste
todas_las_pestanas = pd.read_excel(nombre_archivo, sheet_name=None)

# 3. Unimos las tres pestañas en una sola tabla gigante
lista_tablas = []
for nombre_pestana, df_pestana in todas_las_pestanas.items():
    df_pestana['Pestaña_Equipo'] = nombre_pestana
    lista_tablas.append(df_pestana)

df_completo = pd.concat(lista_tablas, ignore_index=True)

print(f"\n¡Listo! Se unieron {len(todas_las_pestanas)} pestañas en total.")
print("Pestañas encontradas:", list(todas_las_pestanas.keys()))

# 4. Mostramos la tabla completa con todos los equipos
display(df_completo)

Por favor, selecciona tu archivo Excel (.xlsx):


Saving partidos_asistencia_y_precios_2025 OFICIAL.xlsx to partidos_asistencia_y_precios_2025 OFICIAL (3).xlsx

¡Listo! Se unieron 3 pestañas en total.
Pestañas encontradas: ['Colo-Colo', 'Universidad de Chile', 'Universidad Católica']


,Partidos y Asistencia de Local - Colo-Colo,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Pestaña_Equipo,Partidos y Asistencia de Local - Universidad de Chile,Unnamed: 12,Partidos y Asistencia de Local - Universidad Católica
0,Campeonato Nacional 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Colo-Colo,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Colo-Colo,NaN,NaN,NaN
2,Fecha N°,Fecha,Hora,Rival,Estadio,Goles Local,Goles Visita,Resultado,Aforo Autorizado,Público Controlado,% Ocupación,Precio Más Bajo (CLP),Colo-Colo,NaN,NaN,NaN
3,Fecha 2,2025-02-24 00:00:00,20.00,O'Higgins,Estadio Monumental,0,1,Derrota,35000,NaN,-,NaN,Colo-Colo,NaN,NaN,NaN
4,Fecha 4,2025-02-09 00:00:00,20.30,Everton,Estadio Monumental,2,0,Victoria,35000,NaN,-,NaN,Colo-Colo,NaN,NaN,NaN
5,Fecha 6,2025-03-27 00:00:00,18.15,Palestino,Estadio Monumental,1,1,Empate,35000,NaN,-,NaN,Colo-Colo,NaN,NaN,NaN
6,Fecha 9,2025-04-26 00:00:00,20.00,Coquimbo Unido,Estadio Monumental,2,0,Victoria,42000,NaN,-,NaN,Colo-Colo,NaN,NaN,NaN
7,Fecha 11,2025-05-19 00:00:00,18.00,Ñublense,Estadio Monumental,2,2,Empate,35000,NaN,-,NaN,Colo-Colo,NaN,NaN,NaN
8,Fecha 12,2025-05-24 00:00:00,17.30,Unión Española,Estadio Monumental,4,1,Victoria,35000,NaN,-,NaN,Colo-Colo,NaN,NaN,NaN
9,Fecha 14,2025-06-17 00:00:00,18.00,Cobresal,Estadio Monumental,4,0,Victoria,35000,NaN,-,NaN,Colo-Colo,NaN,NaN,NaN


In [ ]:
import altair as alt
import pandas as pd

# 1. Definimos los equipos grandes para identificar los clásicos automáticamente
equipos_grandes = {'Colo-Colo', 'Universidad de Chile', 'Universidad Católica', 'U. de Chile', 'U. Católica', 'Colo Colo'}

def determinar_tipo_partido(row):
    equipo_local = str(row['Equipo']).strip()
    rival = str(row['Rival']).strip()

    # Si el rival es uno de los otros equipos grandes, es un Clásico
    es_rival_grande = any(grande in rival for grande in equipos_grandes if grande.lower() != equipo_local.lower())

    if es_rival_grande:
        return 'Clásico'
    return 'Regular'

# Aplicamos la identificación del tipo de partido al DataFrame limpio
df_limpio['Tipo de Partido'] = df_limpio.apply(determinar_tipo_partido, axis=1)

# 2. Función actualizada para incluir la diferenciación de Clásicos usando círculos
def crear_grafico_club_con_clasicos(df_club, nombre_club, color_base):
    # Configuramos una escala de colores para destacar los Clásicos
    color_scale = alt.Scale(
        domain=['Regular', 'Clásico'],
        range=[color_base, '#D32F2F'] # Rojo intenso para destacar los clásicos
    )

    # Usamos mark_circle para asegurar que todos los marcadores sean círculos perfectos
    return alt.Chart(df_club).mark_circle(opacity=0.85).encode(
        x=alt.X('Fecha:T', title='Fecha del Partido'),
        y=alt.Y('Hora:O', title='Hora de Inicio'),
        color=alt.Color('Tipo de Partido:N', scale=color_scale, title='Tipo de Partido'),
        size=alt.Size('Aforo Autorizado:Q', title='Aforo Autorizado', scale=alt.Scale(range=[100, 800])),
        tooltip=['Fecha N°', 'Fecha', 'Hora', 'Rival', 'Estadio', 'Aforo Autorizado', 'Tipo de Partido']
    ).properties(
        width=700,
        height=220,
        title=f'Partidos de {nombre_club} (2025) - Clásicos vs Regulares'
    ).interactive()

# 3. Separamos el DataFrame actualizado por equipo
df_cc_actualizado = df_limpio[df_limpio['Equipo'] == 'Colo-Colo']
df_uch_actualizado = df_limpio[df_limpio['Equipo'] == 'Universidad de Chile']
df_uc_actualizado = df_limpio[df_limpio['Equipo'] == 'Universidad Católica']

# 4. Generamos los gráficos interactivos independientes
grafico_cc = crear_grafico_club_con_clasicos(df_cc_actualizado, 'Colo-Colo', '#4A4A4A')
grafico_uch = crear_grafico_club_con_clasicos(df_uch_actualizado, 'Universidad de Chile', '#0F4C81')
grafico_uc = crear_grafico_club_con_clasicos(df_uc_actualizado, 'Universidad Católica', '#00A4E4')

# Mostramos la visualización unificada en cascada vertical
graficos_con_clasicos = alt.vconcat(grafico_cc, grafico_uch, grafico_uc).resolve_scale(
    color='independent'
)

display(graficos_con_clasicos)

alt.VConcatChart(...)

In [ ]:
# Guarda el gráfico interactivo en tu computador como HTML
grafico.save('visualizacion.html')
print("¡Archivo HTML guardado con éxito!")

¡Archivo HTML guardado con éxito!
